In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# 크롬 옵션 설정
options = Options()
options.add_experimental_option("detach", True)
options.add_argument("disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0")

driver = webdriver.Chrome(options=options)
results = []

# ✅ 태그 코드 및 이름 (락/메탈)
base_tag = "GR0004"
tag_name = "락/메탈"

# ✅ 페이지 순회 (1 ~ 20)
for page in range(1, 21):
    url = f"https://www.genie.co.kr/playlist/tags?tags={base_tag}%7C%7C{tag_name}&sortOrd=RDD&pg={page}"
    print(f"📄 {page} 페이지 수집 중: {url}")
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select("ul.md_playlist > li")

    for item in items:
        try:
            onclick_attr = item.select_one(".title a")["onclick"]
            playlist_id = re.search(r"playlistLogNDetailView\('(\d+)'", onclick_attr).group(1)
            title = item.select_one(".title a").get_text(strip=True)
            dj_name = item.select_one(".artist").get_text(strip=True)
            tags = [t.get_text(strip=True).lstrip("#") for t in item.select(".tag a")]
            tags_str = ", ".join(tags)
            song_count = re.search(r"\d+", item.select_one("li.num").text).group()
            date_text = item.select_one("li.date").get_text(strip=True)

            results.append({
                "playlist_id": playlist_id,
                "title": title,
                "tags": tags_str,
                "dj_name": dj_name,
                "song_count": int(song_count),
                "date": date_text
            })
        except Exception as e:
            print("❗ 항목 스킵:", e)
            continue

driver.quit()

# 저장
df = pd.DataFrame(results)
df.to_csv("genie_tagpage_summary_락메탈.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ '락/메탈' 20페이지 플레이리스트 수집 완료: {len(df)}개 저장됨")


📄 1 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=1
📄 2 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=2
📄 3 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=3
📄 4 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=4
📄 5 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=5
📄 6 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=6
📄 7 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=7
📄 8 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=8
📄 9 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=9
📄 10 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&pg=10
📄 11 페이지 수집 중: https://www.genie.co.kr/playlist/tags?tags=GR0004%7C%7C락/메탈&sortOrd=RDD&p

In [17]:
df

,playlist_id,title,tags,dj_name,song_count,date
0,17584,자연 풍경과 함께하는 드라이브 뮤직,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신...",DJ 베가스,30,2025.05.23
1,15162,genie pick📍 오늘의 ROCK 디깅 🎸,"디깅 업, DJ 지니, 락/메탈, 휴가/여행, 외출, 기분전환, 힘찬, 신나는, 2...",DJ 지니,100,2025.05.23
2,17589,외로운 밤을 달래줄 감성 록,"레이블 PICK, 드림어스, 인디, 락/메탈, 집, 휴식, 밤, 위로, 잔잔한, 스...",드림어스,30,2025.05.22
3,17586,"여름과 청춘, 그 찬란한 순간들","오늘의선곡, DJ 지니, 락/메탈, 외출, 휴가/여행, 거리, 기분전환, 힘찬, 밝...",DJ 지니,40,2025.05.20
4,17558,Maybe tomorrow☁ 희망찬 내일을 기다리는 노래,"레이블 PICK, 스타DJ, 락/메탈, 힘찬, 위로, 스페셜추천, 스타플레이리스트",스타DJ,14,2025.05.07
...,...,...,...,...,...,...
969,48,한국인 입맛 자극하는 얼터너티브/포스트그런지,"오늘의선곡, DJ 樂PD, 락/메탈, POP, 드라이브, 오후, 해외음악",DJ 樂PD,41,2012.01.20
970,36,진정 좋은 음악이란 무엇인가 [루비살롱 레코드],"오늘의선곡, DJ 지니, 가요, 락/메탈, 출/퇴근길, 드라이브, 힘찬, 분노, 신...",DJ 지니,27,2012.01.05
971,30,락 필드를 호령한 여장부 연대기 (해외편),"오늘의선곡, DJ 樂PD, 락/메탈, POP, 해외음악",DJ 樂PD,23,2012.01.03
972,1,아름다운 마초들이 메탈로 여심을 흔들다!,"오늘의선곡, DJ 樂PD, 락/메탈, 드라이브, 오후, 80년대, 해외음악",DJ 樂PD,19,2012.01.03


In [18]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

# 1. playlist_id + tags 정보 불러오기 (락/메탈)
summary_df = pd.read_csv("genie_tagpage_summary_락메탈.csv")  # playlist_id, tags 포함
playlist_ids = summary_df['playlist_id'].tolist()

# 2. 크롬 설정
options = Options()
options.add_experimental_option("detach", True)
options.add_argument("disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0")

driver = webdriver.Chrome(options=options)

detailed_data = []

# 3. playlist 상세페이지 크롤링
for i, pid in enumerate(playlist_ids):
    url = f"https://www.genie.co.kr/playlist/detailView?plmSeq={pid}"
    print(f"🔍 {i+1}/{len(playlist_ids)}: {url}")
    driver.get(url)
    time.sleep(2)

    # 곡 목록 수집
    rows = driver.find_elements(By.CSS_SELECTOR, "table.list-wrap tbody tr")
    for row in rows:
        try:
            title = row.find_element(By.CSS_SELECTOR, "td.info a.title").text.strip().replace("TITLE", "").strip()
            artist = row.find_element(By.CSS_SELECTOR, "td.info a.artist").text.strip()
            album = row.find_element(By.CSS_SELECTOR, "td.info a.albumtitle").text.strip()

            detailed_data.append({
                "playlist_id": pid,
                "음원제목": title,
                "가수": artist,
                "앨범": album
            })
        except:
            continue

driver.quit()

# 4. DataFrame 생성 및 연관태그 병합
df_detail = pd.DataFrame(detailed_data)

# playlist_id 기준으로 연관태그 붙이기
df = df_detail.merge(summary_df[["playlist_id", "tags"]], on="playlist_id", how="left")

# 장르 추가
df["장르"] = "락/메탈"

# 컬럼 순서 변경 및 이름 정리
df = df[["장르", "음원제목", "가수", "앨범", "tags"]]
df = df.rename(columns={"tags": "연관태그"})

# 저장
df.to_csv("genie_락메탈_음원_연관태그포함.csv", index=False, encoding="utf-8-sig")
print("✅ 락/메탈 상세 곡 정보 저장 완료:", len(df), "곡")

🔍 1/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17584
🔍 2/974: https://www.genie.co.kr/playlist/detailView?plmSeq=15162
🔍 3/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17589
🔍 4/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17586
🔍 5/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17558
🔍 6/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17514
🔍 7/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17505
🔍 8/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17424
🔍 9/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17414
🔍 10/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17380
🔍 11/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17374
🔍 12/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17371
🔍 13/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17299
🔍 14/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17286
🔍 15/974: https://www.genie.co.kr/playlist/detailView?plmSeq=17287
🔍 16

In [19]:
df

,장르,음원제목,가수,앨범,연관태그
0,락/메탈,Spike Island,Pulp,Spike Island,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신..."
1,락/메탈,For Sure,Future Islands,As Long As You Are,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신..."
2,락/메탈,Tropic Morning News,The National,First Two Pages of Frankenstein,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신..."
3,락/메탈,Got to Have Love,Pulp,Got to Have Love,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신..."
4,락/메탈,First Time,Lucy Dacus,Home Video,"레이블 PICK, DJ 베가스, 락/메탈, POP, 거리, 기쁨, 힘찬, 밝은, 신..."
...,...,...,...,...,...
24625,락/메탈,집으로,WINNER,HOLIDAY,"돌비애트모스, DJ 지니, 락/메탈, 출/퇴근길, 운동, 기분전환, 힘찬, 신나는,..."
24626,락/메탈,네버랜드를 떠나며,투모로우바이투게더,이름의 장: TEMPTATION,"돌비애트모스, DJ 지니, 락/메탈, 출/퇴근길, 운동, 기분전환, 힘찬, 신나는,..."
24627,락/메탈,LIAR,i-dle (아이들),I NEVER DIE,"돌비애트모스, DJ 지니, 락/메탈, 출/퇴근길, 운동, 기분전환, 힘찬, 신나는,..."
24628,락/메탈,Folks,LØREN,Put Up a Fight,"돌비애트모스, DJ 지니, 락/메탈, 출/퇴근길, 운동, 기분전환, 힘찬, 신나는,..."
